In [93]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

In [94]:
from catboost import CatBoostRegressor

TRAIN_PATH = r"C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\train.csv"
TEST_PATH = r"C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\test.csv"
SAMPLE_SUB_PATH = r"C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\sample submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Sample shape:", sample_sub.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

Train shape: (78772, 47)
Test shape : (42422, 20)
Sample shape: (42422, 3)

Train columns:
['Id', 'match_id', 'date', 'gender', 'team', 'opponent', 'is_home', 'neutral', 'tournament', 'venue_country', 'team_goals', 'opp_goals', 'team_points_last5', 'opp_points_last5', 'points_last5_diff', 'team_gd_last5', 'opp_gd_last5', 'gd_last5_diff', 'h2h_points_last5', 'h2h_gd_last5', 'days_since_last_match_team', 'days_since_last_match_opp', 'team_points_last10', 'opp_points_last10', 'team_avg_goals_last5', 'team_avg_conceded_last5', 'opp_avg_goals_last5', 'opp_avg_conceded_last5', 'team_win_rate_last10', 'opp_win_rate_last10', 'elo_team', 'elo_opponent', 'rank_team', 'rank_opponent', 'rank_diff', 'rank_missing_team', 'rank_missing_opp', 'confederation_team', 'confederation_opp', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue']

Test columns:
['Id', 'match_id', 'date', 'gender', 

Data Understanading

In [95]:
missing_train = pd.DataFrame({
    "train_missing_count": train.isna().sum(),
    "train_missing_pct": (train.isna().sum() / len(train)) * 100
})

missing_test = pd.DataFrame({
    "test_missing_count": test.isna().sum(),
    "test_missing_pct": (test.isna().sum() / len(test)) * 100
})

missing_summary = missing_train.join(missing_test, how="outer").fillna(0)
missing_summary = missing_summary.sort_values("test_missing_pct", ascending=False)

display(missing_summary)

,train_missing_count,train_missing_pct,test_missing_count,test_missing_pct
distance_travel_opp,30392,38.582237,16975.0,40.014615
distance_travel_team,30392,38.582237,16975.0,40.014615
gdp_per_capita_team,25957,32.952064,14664.0,34.566970
gdp_per_capita_opp,25957,32.952064,14664.0,34.566970
altitude_venue,20554,26.093028,10906.0,25.708359
temperature_venue,17074,21.675215,5594.0,13.186554
population_team,14872,18.879805,3372.0,7.948706
population_opp,14872,18.879805,3372.0,7.948706
days_since_last_match_team,294,0.373229,0.0,0.000000
date,0,0.000000,0.0,0.000000


In [112]:
fi_team = pd.DataFrame({
    "feature": final_features if "final_features" in globals() else base_features,
    "importance": team_model.get_feature_importance()
}).sort_values("importance", ascending=False)


ValueError: All arrays must be of the same length

In [113]:
fi_opp = pd.DataFrame({
    "feature": final_features if "final_features" in globals() else base_features,
    "importance": opp_model.get_feature_importance()
}).sort_values("importance", ascending=False)


ValueError: All arrays must be of the same length

In [97]:
fi_merge = fi_team.merge(fi_opp, on="feature", suffixes=("_team", "_opp"))
fi_merge["importance_avg"] = (fi_merge["importance_team"] + fi_merge["importance_opp"]) / 2
fi_merge = fi_merge.sort_values("importance_avg", ascending=False)

print("=== TOP 30 COMBINED FEATURE IMPORTANCE ===")
display(fi_merge.head(30))

=== TOP 30 COMBINED FEATURE IMPORTANCE ===


,feature,importance_team,importance_opp,importance_avg
0,opponent,22.105972,17.371212,19.738592
1,team,17.756345,21.357932,19.557138
2,gender,9.146062,8.950943,9.048503
3,population_opp,6.771504,4.570598,5.671051
4,is_home,5.234334,5.501185,5.367759
9,population_team,4.421795,5.866101,5.143948
5,tournament,4.967453,4.729646,4.848549
7,confederation_team,4.574944,5.076685,4.825814
6,confederation_opp,4.962608,4.039546,4.501077
8,gdp_per_capita_opp,4.546264,2.558369,3.552317


In [98]:
final_features = [
    "opponent",
    "team",
    "gender",
    "population_opp",
    "population_team",
    "is_home",
    "tournament",
    "confederation_team",
    "confederation_opp",
    "gdp_per_capita_opp",
    "gdp_per_capita_team",
    "venue_country",
    "distance_travel_team",
    "distance_travel_opp",
    "altitude_venue",
    "temperature_venue",
    "day",
    "month",
    "year"
]

categorical_features = [
    col for col in final_features
    if train_modern[col].dtype == "object"
]

numerical_features = [
    col for col in final_features
    if col not in categorical_features
]

print("Jumlah final features:", len(final_features))
print(final_features)

print("\nCategorical:")
print(categorical_features)

print("\nNumerical:")
print(numerical_features)

Jumlah final features: 19
['opponent', 'team', 'gender', 'population_opp', 'population_team', 'is_home', 'tournament', 'confederation_team', 'confederation_opp', 'gdp_per_capita_opp', 'gdp_per_capita_team', 'venue_country', 'distance_travel_team', 'distance_travel_opp', 'altitude_venue', 'temperature_venue', 'day', 'month', 'year']

Categorical:
['opponent', 'team', 'gender', 'tournament', 'confederation_team', 'confederation_opp', 'venue_country']

Numerical:
['population_opp', 'population_team', 'is_home', 'gdp_per_capita_opp', 'gdp_per_capita_team', 'distance_travel_team', 'distance_travel_opp', 'altitude_venue', 'temperature_venue', 'day', 'month', 'year']


Common Feature

In [99]:
train_cols = set(train.columns)
test_cols = set(test.columns)

common_cols = sorted(list(train_cols.intersection(test_cols)))
train_only_cols = sorted(list(train_cols - test_cols))
test_only_cols = sorted(list(test_cols - train_cols))

print("Jumlah common columns :", len(common_cols))
print("Jumlah train-only cols:", len(train_only_cols))
print("Jumlah test-only cols :", len(test_only_cols))

print("\nCommon columns:")
print(common_cols)

print("\nTrain-only columns:")
print(train_only_cols)

Jumlah common columns : 20
Jumlah train-only cols: 27
Jumlah test-only cols : 0

Common columns:
['Id', 'altitude_venue', 'confederation_opp', 'confederation_team', 'date', 'distance_travel_opp', 'distance_travel_team', 'gdp_per_capita_opp', 'gdp_per_capita_team', 'gender', 'is_home', 'match_id', 'neutral', 'opponent', 'population_opp', 'population_team', 'team', 'temperature_venue', 'tournament', 'venue_country']

Train-only columns:
['days_since_last_match_opp', 'days_since_last_match_team', 'elo_opponent', 'elo_team', 'gd_last5_diff', 'h2h_gd_last5', 'h2h_points_last5', 'opp_avg_conceded_last5', 'opp_avg_goals_last5', 'opp_gd_last5', 'opp_goals', 'opp_points_last10', 'opp_points_last5', 'opp_win_rate_last10', 'points_last5_diff', 'rank_diff', 'rank_missing_opp', 'rank_missing_team', 'rank_opponent', 'rank_team', 'team_avg_conceded_last5', 'team_avg_goals_last5', 'team_gd_last5', 'team_goals', 'team_points_last10', 'team_points_last5', 'team_win_rate_last10']


AW MAE Baseline

In [100]:
TOURNAMENT_WEIGHTS = {
    "FIFA World Cup": 2.00,
    "AFC Asian Cup": 1.80,
    "African Cup of Nations": 1.80,
    "UEFA Euro": 1.90,
    "Copa América": 1.90,
    "Confederations Cup": 1.70,
    "Olympic Games": 1.50,
    "Friendly": 0.96
}

DEFAULT_WEIGHT = 1.20

def get_match_outcome(team_goals, opp_goals):
    if team_goals > opp_goals:
        return 1
    elif team_goals < opp_goals:
        return -1
    return 0

def compute_match_loss(y_true_team, y_true_opp, y_pred_team, y_pred_opp, tournament_name):
    # pastikan integer non-negatif
    y_pred_team = max(0, int(round(y_pred_team)))
    y_pred_opp = max(0, int(round(y_pred_opp)))

    # 1. Base MAE
    mae = (abs(y_true_team - y_pred_team) + abs(y_true_opp - y_pred_opp)) / 2

    # 2. Penalty
    exact = int((y_true_team == y_pred_team) and (y_true_opp == y_pred_opp))
    outcome = int(get_match_outcome(y_true_team, y_true_opp) == get_match_outcome(y_pred_team, y_pred_opp))
    gd = int((y_true_team - y_true_opp) == (y_pred_team - y_pred_opp))

    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)

    # 3. Outcome multiplier
    multiplier = 1.0 if outcome == 1 else 1.5

    # 4. Non-linear scaling
    raw_loss = mae + penalty
    loss = (raw_loss * multiplier) ** 1.5

    # 5. Tournament weighting
    weight = TOURNAMENT_WEIGHTS.get(tournament_name, DEFAULT_WEIGHT)

    return loss, weight

def aw_mae_score(df_eval, true_team_col, true_opp_col, pred_team_col, pred_opp_col, tournament_col="tournament"):
    weighted_losses = []
    weights = []

    for _, row in df_eval.iterrows():
        loss, weight = compute_match_loss(
            row[true_team_col],
            row[true_opp_col],
            row[pred_team_col],
            row[pred_opp_col],
            row[tournament_col]
        )
        weighted_losses.append(loss * weight)
        weights.append(weight)

    return np.sum(weighted_losses) / np.sum(weights)

Prepro

In [101]:
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

for df in [train, test]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["decade"] = (df["year"] // 10) * 10

print(train[["date", "year", "month", "dayofweek", "decade"]].head())

        date  year  month  dayofweek  decade
0 1872-11-30  1872     11          5    1870
1 1872-11-30  1872     11          5    1870
2 1873-03-08  1873      3          5    1870
3 1873-03-08  1873      3          5    1870
4 1874-03-07  1874      3          5    1870


In [102]:
drop_cols = ["Id", "match_id", "date", "team_goals", "opp_goals"]

base_features = [col for col in test.columns if col not in ["Id", "match_id", "date"]]
base_features += ["year", "month", "day", "dayofweek", "is_weekend", "decade"]

# pastikan cuma ambil yang ada di train juga
base_features = [col for col in base_features if col in train.columns]

print("Jumlah feature baseline:", len(base_features))
print(base_features)

categorical_features = [
    col for col in base_features
    if train[col].dtype == "object"
]

numerical_features = [
    col for col in base_features
    if col not in categorical_features
]

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Jumlah feature baseline: 29
['gender', 'team', 'opponent', 'is_home', 'neutral', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']

Categorical features:
['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']

Numerical features:
['is_home', 'neutral', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']


In [103]:
exclude_cols = ["Id", "match_id", "date", "team_goals", "opp_goals"]

base_features = [col for col in train.columns if col in test.columns and col not in exclude_cols]

# tambahkan fitur turunan tanggal kalau memang belum ada
extra_date_features = ["year", "month", "day", "dayofweek", "is_weekend", "decade"]
for col in extra_date_features:
    if col in train.columns and col in test.columns and col not in base_features:
        base_features.append(col)

# pastikan unique, urutan tetap aman
base_features = list(dict.fromkeys(base_features))

print("Jumlah feature baseline:", len(base_features))
print(base_features)

categorical_features = [col for col in base_features if train[col].dtype == "object"]
numerical_features = [col for col in base_features if col not in categorical_features]

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

# cek duplikat nama kolom
dup_cols = pd.Series(base_features)[pd.Series(base_features).duplicated()].tolist()
print("\nDuplicate feature names:", dup_cols)

Jumlah feature baseline: 23
['gender', 'team', 'opponent', 'is_home', 'neutral', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']

Categorical features:
['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']

Numerical features:
['is_home', 'neutral', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']

Duplicate feature names: []


In [104]:
MODERN_YEAR = 2000

train_modern = train[train["year"] >= MODERN_YEAR].copy().reset_index(drop=True)

print("Original train shape :", train.shape)
print("Modern train shape   :", train_modern.shape)
print("Year min-max modern  :", train_modern["year"].min(), "-", train_modern["year"].max())

Original train shape : (78772, 53)
Modern train shape   : (27560, 53)
Year min-max modern  : 2000 - 2011


In [105]:
exclude_cols = ["Id", "match_id", "date", "team_goals", "opp_goals"]

keep_cols = []
impute_cols = []
drop_cols_missing = []

for col in base_features:
    test_missing_pct = missing_summary.loc[col, "test_missing_pct"] if col in missing_summary.index else 0
    
    if test_missing_pct > 50:
        drop_cols_missing.append(col)
    elif test_missing_pct > 0:
        impute_cols.append(col)
    else:
        keep_cols.append(col)

print("=== KEEP COLS ===")
print(keep_cols)

print("\n=== IMPUTE COLS ===")
print(impute_cols)

print("\n=== DROP COLS (missing test > 50%) ===")
print(drop_cols_missing)

=== KEEP COLS ===
['gender', 'team', 'opponent', 'is_home', 'neutral', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']

=== IMPUTE COLS ===
['population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue']

=== DROP COLS (missing test > 50%) ===
[]


In [106]:
final_features = [col for col in base_features if col not in drop_cols_missing]

categorical_features = [
    col for col in final_features
    if train_modern[col].dtype == "object"
]

numerical_features = [
    col for col in final_features
    if col not in categorical_features
]

print("Jumlah final features:", len(final_features))
print(final_features)

print("\nCategorical:")
print(categorical_features)

print("\nNumerical:")
print(numerical_features)

Jumlah final features: 23
['gender', 'team', 'opponent', 'is_home', 'neutral', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']

Categorical:
['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']

Numerical:
['is_home', 'neutral', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']


In [107]:
train_modern_imp = train_modern.copy()
test_imp = test.copy()

for col in categorical_features:
    train_modern_imp[col] = train_modern_imp[col].fillna("Unknown")
    test_imp[col] = test_imp[col].fillna("Unknown")

for col in numerical_features:
    med = train_modern_imp[col].median()
    train_modern_imp[col] = train_modern_imp[col].fillna(med)
    test_imp[col] = test_imp[col].fillna(med)

print("Imputasi selesai.")
print("Train missing remaining:", train_modern_imp[final_features].isna().sum().sum())
print("Test missing remaining:", test_imp[final_features].isna().sum().sum())

Imputasi selesai.
Train missing remaining: 0
Test missing remaining: 0


Split

In [109]:
train_modern_imp = train_modern_imp.sort_values("date").reset_index(drop=True)

split_date = train_modern_imp["date"].quantile(0.85)

train_part = train_modern_imp[train_modern_imp["date"] < split_date].copy()
valid_part = train_modern_imp[train_modern_imp["date"] >= split_date].copy()

X_train = train_part[final_features].copy()
X_valid = valid_part[final_features].copy()

y_train_team = train_part["team_goals"].copy()
y_train_opp = train_part["opp_goals"].copy()

y_valid_team = valid_part["team_goals"].copy()
y_valid_opp = valid_part["opp_goals"].copy()

print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)

Train shape: (23380, 23)
Valid shape: (4180, 23)


training `team_goals`

In [114]:
team_model = CatBoostRegressor(
    iterations=700,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100
)

team_model.fit(
    X_train,
    y_train_team,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid_team),
    use_best_model=True
)

0:	learn: 1.1578756	test: 1.1296762	best: 1.1296762 (0)	total: 28.2ms	remaining: 19.7s
100:	learn: 1.0413369	test: 1.0485458	best: 1.0485458 (100)	total: 3.91s	remaining: 23.2s
200:	learn: 1.0142652	test: 1.0364971	best: 1.0364971 (200)	total: 8.54s	remaining: 21.2s
300:	learn: 0.9935024	test: 1.0312234	best: 1.0311673 (299)	total: 11.5s	remaining: 15.2s
400:	learn: 0.9800404	test: 1.0298676	best: 1.0298435 (399)	total: 14.5s	remaining: 10.8s
500:	learn: 0.9660306	test: 1.0278083	best: 1.0277255 (498)	total: 18.1s	remaining: 7.18s
600:	learn: 0.9563137	test: 1.0273004	best: 1.0270501 (568)	total: 20.9s	remaining: 3.45s
699:	learn: 0.9460887	test: 1.0268470	best: 1.0267991 (684)	total: 23.8s	remaining: 0us

bestTest = 1.026799112
bestIteration = 684

Shrink model to first 685 iterations.


training `opp_goals`

In [115]:
opp_model = CatBoostRegressor(
    iterations=700,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100
)

opp_model.fit(
    X_train,
    y_train_opp,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid_opp),
    use_best_model=True
)

0:	learn: 1.1584722	test: 1.1302504	best: 1.1302504 (0)	total: 25.7ms	remaining: 18s
100:	learn: 1.0419953	test: 1.0483878	best: 1.0483878 (100)	total: 2.71s	remaining: 16.1s
200:	learn: 1.0172968	test: 1.0393712	best: 1.0393712 (200)	total: 5.63s	remaining: 14s
300:	learn: 0.9949870	test: 1.0323075	best: 1.0321646 (288)	total: 8.82s	remaining: 11.7s
400:	learn: 0.9794000	test: 1.0291465	best: 1.0291465 (400)	total: 12.6s	remaining: 9.37s
500:	learn: 0.9667936	test: 1.0279749	best: 1.0278191 (490)	total: 15.5s	remaining: 6.17s
600:	learn: 0.9554773	test: 1.0277455	best: 1.0276027 (587)	total: 18.6s	remaining: 3.06s
699:	learn: 0.9453685	test: 1.0269348	best: 1.0267823 (687)	total: 22.1s	remaining: 0us

bestTest = 1.026782318
bestIteration = 687

Shrink model to first 688 iterations.


eval aw-mae

In [74]:
valid_pred_team = team_model.predict(X_valid)
valid_pred_opp = opp_model.predict(X_valid)

print("Raw prediction stats:")
print("team -> min:", np.min(valid_pred_team), "max:", np.max(valid_pred_team), "mean:", np.mean(valid_pred_team))
print("opp  -> min:", np.min(valid_pred_opp), "max:", np.max(valid_pred_opp), "mean:", np.mean(valid_pred_opp))

valid_eval = valid_part.copy()
valid_eval["pred_team_goals"] = np.round(np.clip(valid_pred_team, 0, 5)).astype(int)
valid_eval["pred_opp_goals"] = np.round(np.clip(valid_pred_opp, 0, 5)).astype(int)

awmae = aw_mae_score(
    valid_eval,
    true_team_col="team_goals",
    true_opp_col="opp_goals",
    pred_team_col="pred_team_goals",
    pred_opp_col="pred_opp_goals",
    tournament_col="tournament"
)

mae_team = mean_absolute_error(valid_eval["team_goals"], valid_eval["pred_team_goals"])
mae_opp = mean_absolute_error(valid_eval["opp_goals"], valid_eval["pred_opp_goals"])
overall_mae = (
    (valid_eval["team_goals"] - valid_eval["pred_team_goals"]).abs() +
    (valid_eval["opp_goals"] - valid_eval["pred_opp_goals"]).abs()
).mean() / 2

print("\nValidation MAE team :", mae_team)
print("Validation MAE opp  :", mae_opp)
print("Validation Overall MAE:", overall_mae)
print("Validation AW-MAE   :", awmae)

display(valid_eval[[
    "date", "team", "opponent", "tournament",
    "team_goals", "opp_goals",
    "pred_team_goals", "pred_opp_goals"
]].head(20))

Raw prediction stats:
team -> min: -0.5582626291583612 max: 6.714035632842133 mean: 1.2708123397419993
opp  -> min: -0.6697198347065811 max: 7.492752030532508 mean: 1.2575107088582387

Validation MAE team : 1.0038277511961722
Validation MAE opp  : 1.0088516746411482
Validation Overall MAE: 1.0063397129186602
Validation AW-MAE   : 3.167358137669751


,date,team,opponent,tournament,team_goals,opp_goals,pred_team_goals,pred_opp_goals
23380,2009-11-18,Germany,Ivory Coast,Friendly,2,2,2,1
23381,2009-11-18,Portugal,Bosnia and Herzegovina,FIFA World Cup qualification,1,0,1,1
23382,2009-11-18,Bosnia and Herzegovina,Portugal,FIFA World Cup qualification,0,1,1,1
23383,2009-11-18,Spain,Austria,Friendly,5,1,1,1
23384,2009-11-18,United States,Denmark,Friendly,1,3,1,1
23385,2009-11-18,Egypt,Algeria,FIFA World Cup qualification,0,1,1,1
23386,2009-11-18,Algeria,Egypt,FIFA World Cup qualification,1,0,1,1
23387,2009-11-18,Ghana,Angola,Friendly,0,0,1,1
23388,2009-11-18,Angola,Ghana,Friendly,0,0,1,1
23389,2009-11-18,Ivory Coast,Germany,Friendly,2,2,1,2


-----------------

train full model buat submission

In [ ]:
X_full = train[base_features].copy()
y_full_team = train["team_goals"].copy()
y_full_opp = train["opp_goals"].copy()

X_test = test[base_features].copy()

final_team_model = CatBoostRegressor(
    iterations=team_model.get_best_iteration() if team_model.get_best_iteration() is not None else 1200,
    learning_rate=0.03,
    depth=8,
    loss_function="MAE",
    random_seed=42,
    verbose=100
)

final_opp_model = CatBoostRegressor(
    iterations=opp_model.get_best_iteration() if opp_model.get_best_iteration() is not None else 1200,
    learning_rate=0.03,
    depth=8,
    loss_function="MAE",
    random_seed=42,
    verbose=100
)

final_team_model.fit(
    X_full, y_full_team,
    cat_features=categorical_features
)

final_opp_model.fit(
    X_full, y_full_opp,
    cat_features=categorical_features
)

In [ ]:
test_pred_team = final_team_model.predict(X_test)
test_pred_opp = final_opp_model.predict(X_test)

submission = sample_sub.copy()
submission["team_goals"] = np.round(np.clip(test_pred_team, 0, None)).astype(int)
submission["opp_goals"] = np.round(np.clip(test_pred_opp, 0, None)).astype(int)

submission.to_csv("submission_baseline_catboost.csv", index=False)

print(submission.head())
print("\nSaved: submission_baseline_catboost.csv")